[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab09_regular_expressions/lab09_student.ipynb)

# Lab 09 — Regular Expressions for Text Cleaning

### Digital Skills and Business Decisioning Using Python

In Lab 08 you cleaned text with string methods: `.lower()`, `.split()`, `.strip(string.punctuation)`. They work well when you know **exactly** what you're looking for. But a lot of useful questions about text are about *shapes*, not exact strings:

* Find every **number** in a review (a size, a price, a weight).
* Find a customer's **height**, written as `5'4`, `5' 10`, or `5'2"`.
* Replace **any amount** of whitespace, including line breaks, with a single space.
* Count how many **exclamation marks** a customer used, and squash `!!!!!!` into one `!`.

**Regular expressions** (regex for short) are a small language for describing patterns like these. They're built into Python through the `re` module, and they're also what pandas uses under the hood when you clean a text column, so everything in this lab will come back once you start working with DataFrames.

Regex has a reputation for being cryptic. The trick is to learn a **small** set of building blocks well, and to test every pattern on examples you can see. That's what this lab does. You'll learn to:

* find every match of a pattern with **`re.findall()`**,
* describe *kinds* of characters with **character classes** (`\d`, `\w`, `\s`, `[...]`),
* say *how many* with **quantifiers** (`+`, `*`, `?`, `{n,m}`),
* match whole words with **`\b`**, and ignore case with **`re.IGNORECASE`**,
* find one match and pull pieces out of it with **`re.search()`** and **groups**,
* clean text with **`re.sub()`**, and
* turn patterns into **numeric features** you can compare across star ratings.

### Part 1: Your First Pattern

Here's a real review (slightly shortened). Say you want every number in it. With string methods you'd have to loop over the characters and glue digits together yourself. With regex it's one line:

In [ ]:
import re

review = "I'm 5'2\", 110 lbs and bought the xs petite. I'm usually a 0 or 2 and it fits perfectly!"

print(re.findall(r"\d+", review))

`re.findall(pattern, text)` scans the text from left to right and returns a **list** of every piece that matches the pattern. Here the pattern `\d+` means *"one or more digits in a row"*, so `110` comes back as one match instead of three separate digits.

(The `\"` inside the review is how you put a double-quote character inside a string that's already wrapped in double quotes, like the `\n` and `\t` you've seen before.)

### Raw Strings: Why Every Pattern Starts With `r`

Regex uses the backslash `\` a lot, and so do ordinary Python strings (`\n` is a newline, `\t` a tab). To stop Python from interpreting the backslashes before `re` ever sees them, write patterns as **raw strings**, with an `r` right before the opening quote. In a raw string, a backslash is just a backslash.

Run the code cell below:

In [ ]:
print("one\ntwo")
print(r"one\ntwo")

In the normal string, `\n` became a line break. In the raw string, it stayed as the two characters `\` and `n`, which is exactly what `re` needs to receive. **Always write regex patterns as raw strings**, even when a pattern happens to have no backslashes. It's a habit that prevents very confusing bugs.

---

### Part 2: Character Classes, Describing Kinds of Characters

A **character class** matches any **one** character from a group. Python has shortcuts for the groups you'll need most:

| Pattern | Matches one... | Example matches |
|---|---|---|
| `\d` | digit | `0` ... `9` |
| `\w` | "word" character: letter, digit, or underscore | `a`, `Z`, `7`, `_` |
| `\s` | whitespace character: space, tab, newline | `" "`, `\t`, `\n` |
| `.` | any character at all (except a newline) | anything |
| `\D`, `\W`, `\S` | the **opposite** of `\d`, `\w`, `\s` | e.g. `\D` is any non-digit |

Run the code cell below:

In [ ]:
text = "Size 4, $98 & 2 colors!"

print(re.findall(r"\d", text))
print(re.findall(r"\w", text))
print(re.findall(r"\s", text))
print(re.findall(r"\W", text))

Without a quantifier, each pattern matches **one character at a time**, so `\d` returns the digits `4`, `9`, `8`, `2` separately. `\W` (capital W) finds everything that is *not* a word character: the spaces, the comma, `$`, `&`, and `!`. Capital letters flip the meaning of the shortcut.

### Your Own Classes: `[...]`

Square brackets let you list exactly which characters are allowed:

* `[aeiou]` matches one vowel.
* `[a-z]` matches one lowercase letter (a **range**); `[0-9]` is the same as `\d`.
* `[a-zA-Z]` matches one letter of either case: ranges can be combined.
* `[^...]` with a `^` **first** means *"any character except these"*. `[^a-z ]` is anything that isn't a lowercase letter or a space.

Run the code cell below:

In [ ]:
text = "xs, s, m, l, xl"

print(re.findall(r"[xsml]", text))
print(re.findall(r"[^a-z]", text))

### Special Characters Need a Backslash

Some characters have a special meaning in regex: `.` (any character), `$`, `?`, `+`, `*`, `(`, `)`, `[`, `]`, and a few more. To match one of them **literally**, put a backslash in front of it: `\.` matches an actual full stop, `\$` an actual dollar sign.

Run the code cell below and compare the two lines:

In [ ]:
text = "It was $98. Worth it."

print(re.findall(r".", text)[:6])
print(re.findall(r"\.", text))
print(re.findall(r"\$", text))

`.` on its own matches every character (only the first six are shown), while `\.` matches only the two real full stops.

### Practice: Character Classes

**Exercise 1: Every Digit (easy)**

Use `re.findall()` with `\d` to list every single digit in `review`, then print how many there are.

*Expected result:* `['5', '7', '1', '3', '0', '6']` and `6`

In [ ]:
review = "I'm 5'7, 130 lbs and the 6 fits great."

# TODO: find every digit with re.findall and \d, print the list and its length

**Exercise 2: Just the Punctuation (medium)**

Write **one** character class that matches any character that is **not** a letter, a digit, or a space (hint: `[^...]` with `a-z`, `A-Z`, `0-9`, and a space inside). Use it with `re.findall()` on `review` and print the result.

*Expected result:* `['!', '!', ',', '.', '/', '!']`

In [ ]:
review = "Love it!! Soft, warm. 10/10 would buy again!"

# TODO: find every character that is not a letter, digit, or space

---

### Part 3: Quantifiers, Saying How Many

A **quantifier** goes right after a character or class and says how many times it may repeat:

| Quantifier | Means | Example | Matches |
|---|---|---|---|
| `+` | one or more | `\d+` | `4`, `98`, `2024` |
| `*` | zero or more | `so*` | `s`, `so`, `sooo` |
| `?` | zero or one (optional) | `colou?r` | `color`, `colour` |
| `{n}` | exactly `n` | `\d{4}` | `2024` |
| `{n,}` | `n` or more | `o{3,}` | `ooo`, `oooooo` |
| `{n,m}` | between `n` and `m` | `\d{1,2}` | `4`, `12` |

Run the code cell below:

In [ ]:
review = "Sooooo cute!!! The colour is great, the color is true. I love it!"

print(re.findall(r"!+", review))
print(re.findall(r"o{3,}", review))
print(re.findall(r"colou?r", review))
print(re.findall(r"\w+", review))

Each line answers a small question:

* `!+` groups runs of exclamation marks, so `!!!` is one match and the final `!` is another.
* `o{3,}` finds stretched-out words (`Sooooo`): three or more `o`s in a row.
* `colou?r` matches both spellings, because the `u` is optional.
* `\w+` (one or more word characters) is a quick **tokenizer**: it pulls out every word and drops all punctuation. Compare that with the `tokenize()` loop you wrote in Lab 08.

### Combining Pieces: Prices

Patterns are built by putting pieces side by side. A price like `$148.00` or `$98` is: a dollar sign, one or more digits, then *optionally* a full stop and more digits.

In [ ]:
text = "Not worth $148.00, maybe $98. I paid $20 more than the $1 fee."

print(re.findall(r"\$\d+\.?\d*", text))

Read the pattern piece by piece: `\$` a literal dollar sign, `\d+` one or more digits, `\.?` an optional full stop, `\d*` zero or more digits. Look closely at the output, though: `$98.` includes the full stop that ends the sentence, because the pattern allows a dot even when no digits follow. Every pattern is a trade-off between simple and precise, and **checking the output on real examples is the only way to find these edge cases**. If you needed prices exactly right, `\$\d+(?:\.\d\d)?` fixes it, but that uses syntax outside the scope of this course. For counting "does this review mention a price?", the simple version is fine.

> **Quantifiers are greedy.** `+` and `*` grab as much as they possibly can. That's usually what you want (`\d+` takes all of `148`, not just `1`), but combined with `.` it can surprise you: `".+"` on `the "xs" and the "s"` matches `"xs" and the "s"` as a single match, from the first quote to the last. If a pattern seems to swallow too much, a `.+` or `.*` is almost always the reason.

### Practice: Quantifiers

**Exercise 3: Whole Numbers (easy)**

Find every whole number (one or more digits in a row) in `review` and print the list.

*Expected result:* `['5', '4', '120', '2', '4']`

In [ ]:
review = "I'm 5'4 and 120 lbs, usually a size 2 or 4."

# TODO: find every whole number

**Exercise 4: Stretched Words (medium)**

Customers stretch words for emphasis: `sooooo`, `loooove`, `cuuute`. Find every **whole word** that contains a run of three or more `o`, `u`, or `e` letters. The pattern has three parts side by side: `\w*` (any word characters before the run), `[oue]{3,}` (the run itself), and `\w*` again (any word characters after it). Print the result.

*Expected result:* `['sooooo', 'loooove', 'cuuute']`

In [ ]:
review = "this dress is sooooo pretty, i loooove it, so cuuute and good"

# TODO: find the stretched words using \w*[oue]{3,}\w*

---

### Part 4: Whole Words and Ignoring Case

Search for `fit` with a plain pattern and you'll also match it *inside* longer words:

In [ ]:
review = "The fit is great. Fitted waist, and it fits with any outfit."

print(re.findall(r"fit", review))
print(re.findall(r"\bfit\b", review))

The first pattern found `fit` inside `fits` and `outfit` too (but not in `Fitted`, because `F` is uppercase and patterns are case-sensitive). `\b` is a **word boundary**: it doesn't match a character, it matches the *edge* of a word, the point between a word character and a non-word character. Putting `\b` on both sides means "`fit` as a whole word", so only one match is left.

This matters for counting: if you count `fit` without `\b`, every `outfit` gets counted as a comment about fit.

### `re.IGNORECASE`

To match regardless of case, pass **`flags=re.IGNORECASE`** (or its short form `re.I`). It's usually cleaner than writing `[Ff]it` or lowercasing everything first:

In [ ]:
print(re.findall(r"\bfit\w*", review))
print(re.findall(r"\bfit\w*", review, flags=re.IGNORECASE))

`\bfit\w*` means "a word that **starts** with `fit`", so it catches `fit`, `fits`, and (once case is ignored) `Fitted`, but not `outfit`, whose `fit` isn't at the start of a word.

### Anchors: `^` and `$`

`^` matches the **start** of the string and `$` the **end**. Like `\b`, they don't consume characters; they pin the pattern to a position.

(Careful: `^` means "start of the string" when it's at the beginning of a pattern, but "not" when it's the first thing **inside** square brackets. Same symbol, two jobs.)

In [ ]:
titles = ["Love it!", "Beautiful but runs small", "Not what I expected", "love love love"]

for title in titles:
    starts_with_love = bool(re.search(r"^love", title, flags=re.IGNORECASE))
    ends_with_bang = bool(re.search(r"!$", title))
    print(f"{title:<28} starts with love: {starts_with_love!s:<6} ends with !: {ends_with_bang}")

That cell used `re.search()`, which you'll meet properly in the next part. For now, `bool(re.search(...))` just means "is there a match anywhere? `True` or `False`".

### Practice: Whole Words

**Exercise 5: Petite or Not? (medium)**

Count how many times the whole word `petite` appears in `review`, **ignoring case**. It should not count `petites` or `nonpetite`.

*Expected result:* `2`

In [ ]:
review = "Petite sizing is great. The petites run short, but the petite xs fits me. Nonpetite runs long."

# TODO: count whole-word matches of petite, ignoring case

---

### Part 5: Finding One Match and Pulling Pieces Out of It

`re.findall()` gives you all the matches as strings. **`re.search()`** looks for the **first** match only and returns a **match object** with extra information, or `None` if there's no match at all.

In [ ]:
review = "I'm 5'4 and 125 lbs. The small was perfect."

match = re.search(r"\d+ lbs", review)
print(match)
print(match.group())

`.group()` returns the text that matched. Because `re.search()` returns `None` when nothing matches, and `None.group()` crashes with an `AttributeError`, always check the result before using it. Truthiness (Lab 05) makes that easy: a match object counts as `True`, and `None` counts as `False`.

In [ ]:
for review in ["I'm 125 lbs, the small fit.", "Gorgeous color, runs large."]:
    match = re.search(r"\d+ lbs", review)
    if match:
        print("Weight mentioned:", match.group())
    else:
        print("No weight mentioned.")

### Groups: `( )`

Put **parentheses** around part of a pattern, and the match object remembers that part separately as a numbered **group**. `.group(1)` is the text inside the first pair of parentheses, `.group(2)` the second, and so on (`.group()` or `.group(0)` is still the whole match).

This is how you extract structured information from free text. Customers often write their height as feet and inches, like `5'4` or `5' 10"`. The pattern below captures the feet and the inches separately:

In [ ]:
review = "I'm 5' 10\" and usually wear a medium."

match = re.search(r"(\d)'\s?(\d{1,2})", review)
print(match.group())
print(match.group(1))
print(match.group(2))

Piece by piece: `(\d)` one digit for the feet (group 1), `'` a literal apostrophe, `\s?` an optional space, `(\d{1,2})` one or two digits for the inches (group 2). Groups come back as **strings**, so convert them with `int()` before doing maths.

Wrapped in a function, this turns messy text into a number you can analyse. One inch is 2.54 cm and one foot is 12 inches:

In [ ]:
def height_cm(text):
    """Return the first height written as feet'inches in text, in cm, or None."""
    match = re.search(r"(\d)'\s?(\d{1,2})", text)
    if not match:
        return None
    feet = int(match.group(1))
    inches = int(match.group(2))
    return round((feet * 12 + inches) * 2.54)


print(height_cm("I'm 5'4 and 125 lbs"))
print(height_cm("5' 10\", long torso"))
print(height_cm("Great color, runs large"))

### `findall()` With Groups

One behaviour to know: when a pattern has groups, `re.findall()` returns the **groups** instead of the whole match. With one group you get a list of strings; with two or more you get a list of **tuples**.

In [ ]:
text = "My sister is 5'2 and I'm 5'7."

print(re.findall(r"\d'\d{1,2}", text))
print(re.findall(r"(\d)'(\d{1,2})", text))

This is convenient when you want the pieces (tuple unpacking in a `for` loop works nicely: `for feet, inches in re.findall(...)`), and surprising if you just wanted the whole matches. If `findall()` suddenly returns tuples, check your pattern for parentheses.

### Practice: Search and Groups

**Exercise 6: Extract the Size (medium)**

Write a function `ordered_size(text)` that finds the first place where a review says `size` followed by a space and a number (e.g. `size 4`, `size 12`), and **returns the number as an `int`**, or `None` if there isn't one. Use `re.search()`, one group, and `flags=re.IGNORECASE`. Test it on the three reviews below.

*Expected result:* `4`, `12`, `None`

In [ ]:
tests = ["I ordered a size 4 and it fits.", "Size 12 was too big!", "Runs small, order up."]

# TODO: define ordered_size(text), then print ordered_size(t) for each t in tests

---

### Part 6: Cleaning Text With `re.sub()`

**`re.sub(pattern, replacement, text)`** replaces **every** match of a pattern with a replacement string. It's `.replace()` from Lab 02, but for patterns instead of exact text.

Here's a real review exactly as it appears in the dataset, line breaks included (`\r\n` is how Windows writes a line break):

In [ ]:
raw = "Bought both colors in xs and soooooo glad i did.   this dress is gorgeous...\r\n\r\na must have for the summer!!!!!!"
print(raw)

Three regex substitutions clean it up:

In [ ]:
step1 = re.sub(r"\s+", " ", raw)
print(step1)

step2 = re.sub(r"!{2,}", "!", step1)
print(step2)

step3 = re.sub(r"\.{2,}", ".", step2)
print(step3)

* `\s+` → `" "`: any run of whitespace (spaces, tabs, `\r`, `\n`) becomes a single space, so the text sits on one line.
* `!{2,}` → `"!"`: two or more exclamation marks become one.
* `\.{2,}` → `"."`: an ellipsis becomes a single full stop.

Notice the order: each step works on the output of the previous one.

### A `clean_text()` Function

For counting words, you usually want to go further and keep **only letters and spaces**. A negated class does that in one step: replace anything that *isn't* a letter, an apostrophe, or whitespace with a space, then collapse the extra spaces.

In [ ]:
def clean_text(text):
    """Lowercase text, keep only letters, apostrophes and spaces, and collapse whitespace."""
    text = text.lower()
    text = re.sub(r"[^a-z'\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print(clean_text(raw))
print(clean_text("I'm 5'4, 120 lbs; the XS (petite) fits!!"))

Every digit and punctuation mark except the apostrophe is gone, so `i'm` survives while `5'4` falls apart into a stray `'`. That's the kind of small trade-off you'll make every time you clean text. Here it's harmless: when counting words, a lone `'` can be filtered out like any stopword.

Compare with Lab 08: `tokenize()` needed a loop, `.strip()`, and an `if` to do less than this. After `clean_text()`, a plain `.split()` gives you the tokens.

In [ ]:
print(clean_text(raw).split())

### Practice: `re.sub()`

**Exercise 7: Hide the Numbers (easy)**

Before sharing reviews with a colleague, you want to hide every number. Replace every run of digits in `review` with `#` and print the result.

*Expected result:* `I'm #'#, # lbs, size #.`

In [ ]:
review = "I'm 5'6, 140 lbs, size 8."

# TODO: replace every run of digits with "#"

**Exercise 8: Normalize Spacing and Punctuation (medium)**

Write a function `tidy(text)` that:

1. replaces any run of whitespace with a single space,
2. replaces a run of two or more `!` or `?` characters with a single one of the same kind (do `!` and `?` in two separate steps), and
3. removes spaces at the start and end.

Print `tidy(messy)`.

*Expected result:* `Is it lined? Yes! Love it!`

In [ ]:
messy = "   Is it   lined???\n\nYes!!!   Love it!! "

# TODO: define tidy(text) with three steps, then print tidy(messy)

---

### Part 7: From Patterns to Features

Here's where regex starts paying off for analysis. A pattern can turn every review into a **number**, and numbers can be compared across groups. Let's ask two questions of the 600 real reviews from Lab 08:

1. Do happier customers use more **exclamation marks**?
2. Do customers who mention their **height** give different ratings?

First, download and load the file exactly as in Lab 08:

In [ ]:
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/reviews_sample.txt"
urllib.request.urlretrieve(DATA_URL, "reviews_sample.txt")

reviews = []
with open("reviews_sample.txt", encoding="utf-8") as file_object:
    for line in file_object:
        rating, text = line.rstrip("\n").split("\t")
        reviews.append({"rating": int(rating), "text": text})

print(f"Loaded {len(reviews)} reviews.")

Now compute two **features** for every review and store them right in its dictionary:

* `n_exclaims`: how many `!` characters it contains (`len(re.findall(...))`),
* `mentions_height`: whether it contains a height like `5'4` (`bool(re.search(...))`).

In [ ]:
for review in reviews:
    review["n_exclaims"] = len(re.findall(r"!", review["text"]))
    review["mentions_height"] = bool(re.search(r"\d'\s?\d{1,2}", review["text"]))

print(reviews[0])

Then summarise each feature **by star rating**. For each rating from 1 to 5, collect the matching reviews and compute the average number of exclamation marks and the share of reviews that mention a height.

(Averaging `True`/`False` values works because Python treats `True` as `1` and `False` as `0`, so the average of a list of Booleans is the share that are `True`.)

In [ ]:
print(f"{'Rating':<8}{'Reviews':>8}{'Avg !':>8}{'Height %':>10}")

for rating in range(1, 6):
    group = [review for review in reviews if review["rating"] == rating]
    avg_exclaims = sum(review["n_exclaims"] for review in group) / len(group)
    height_share = sum(review["mentions_height"] for review in group) / len(group)
    print(f"{rating:<8}{len(group):>8}{avg_exclaims:>8.2f}{100 * height_share:>9.0f}%")

Look at the two columns:

* **Exclamation marks rise with the rating.** 5-star reviews use roughly three times as many as 1-star reviews. Enthusiasm shows up in punctuation, so `n_exclaims` is a feature worth keeping for later analysis.
* **Height mentions change much less.** Between roughly 1 in 7 and 1 in 5 reviews mention a height at every rating level. Customers give their height to help others judge the fit, whether they liked the item or not. That's also a finding: this feature mostly describes *how* people write reviews, not *how happy* they are.

Keep the sample size in mind: there are only 20 one-star reviews here, so the numbers for low ratings are rough. In the pandas labs you'll repeat this on all ~23,000 reviews, with far less code.

### Practice: Features

**Exercise 9: Do Unhappy Customers Mention Price? (medium)**

Add a feature `mentions_price` to every review: `True` if the text contains the whole word `price`, `pricey`, `expensive`, or `cheap` (ignoring case). One pattern can do it with `|`, which means **or** inside a group: `\b(price|pricey|expensive|cheap)\b`.

Then print the share of reviews mentioning price among **negative** (1-2 stars) and **positive** (4-5 stars) reviews, as percentages with one decimal place.

In [ ]:
# TODO: add review["mentions_price"] to each review, then print the share for negative and positive reviews

**Exercise 10: Clean, Then Count (medium)**

Combine this lab with Lab 08: use `clean_text()` and `.split()` to tokenize every review, count all tokens with a `Counter`, and print the 10 most common words that are **at least 4 letters long** (filter with a list comprehension on `len(word) >= 4` before counting).

Some people use a length filter as a quick substitute for a stopword list. Look at your output: which words in your top 10 would the `STOPWORDS` set from Lab 08 have removed? What does that tell you about the shortcut?

In [ ]:
from collections import Counter

# TODO: clean and split every review, keep words of 4+ letters, count them, print the 10 most common

---

### Recap: Your Regex Toolkit

**Functions**

| Function | What It Does | Example |
|---|---|---|
| `re.findall(p, text)` | List of every match (or of the groups, if `p` has any) | `re.findall(r"\d+", "size 4 or 6")` → `['4', '6']` |
| `re.search(p, text)` | First match as a match object, or `None` | `re.search(r"\d+ lbs", text)` |
| `match.group()` / `.group(1)` | Whole match / text of group 1 | `match.group(1)` |
| `re.sub(p, new, text)` | Replace every match with `new` | `re.sub(r"\s+", " ", text)` |
| `flags=re.IGNORECASE` | Ignore upper/lower case | `re.findall(r"fit", t, flags=re.IGNORECASE)` |

**Pattern building blocks**

| Pattern | Matches |
|---|---|
| `\d` `\w` `\s` | one digit / word character / whitespace character |
| `\D` `\W` `\S` | one character that is **not** a digit / word character / whitespace |
| `.` | any one character (except newline) |
| `[abc]` `[a-z]` | one character from the list / range |
| `[^abc]` | one character **not** in the list |
| `\.` `\$` `\?` `\!` | a literal special character |
| `+` `*` `?` | one or more / zero or more / optional |
| `{n}` `{n,}` `{n,m}` | exactly `n` / `n` or more / between `n` and `m` |
| `\b` | a word boundary (edge of a word) |
| `^` `$` | start / end of the string |
| `( )` | a group, to extract part of a match |
| `a\|b` | `a` or `b` |

The whole lab in one sentence: **a regex describes the *shape* of the text you want, `findall` and `search` find it, `sub` replaces it, and counting matches turns free text into numbers you can compare.**